# relSCAN reproduction (Chemical-Induced Disease relation extraction)
Pipeline: download BC5CDR -> text processing -> relation instances (Co-occurrence / Non-Co-occurrence, 4 forms)
-> 27 features -> SVM + decision tree (J48-like) soft vote -> merge -> evaluate.

In [ ]:
# CELL 1: install (Colab)
!pip -q install spacy scikit-learn
!python -m spacy download en_core_web_sm

In [ ]:
# CELL 2: download + parse BC5CDR (PubTator format)
import os, re, glob, zipfile, urllib.request, collections
import numpy as np

URL = "https://github.com/JHnlp/BioCreative-V-CDR-Corpus/raw/master/CDR_Data.zip"
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/CDR_Data.zip"):
    urllib.request.urlretrieve(URL, "data/CDR_Data.zip")
with zipfile.ZipFile("data/CDR_Data.zip") as z:
    z.extractall("data")

def find_file(keyword):
    # only the gold corpus folder (the zip also contains DNorm / tmChem files that also match "Test")
    return [f for f in glob.glob("data/CDR_Data/CDR.Corpus.v010516/*PubTator.txt") if keyword in os.path.basename(f)][0]

def parse_pubtator(path):
    docs = {}
    raw = open(path, encoding="utf-8").read().strip()
    for block in raw.split("\n\n"):
        pmid, title, abstract, mentions, relations = None, "", "", [], []
        for line in block.split("\n"):
            if not line.strip():
                continue
            if re.match(r"^\d+\|t\|", line):
                pmid, _, title = line.split("|", 2)
            elif re.match(r"^\d+\|a\|", line):
                abstract = line.split("|", 2)[2]
            else:
                p = line.split("\t")
                if len(p) == 4 and p[1] == "CID":
                    relations.append({"chemical": p[2], "disease": p[3]})
                elif len(p) >= 5:
                    mentions.append({"start": int(p[1]), "end": int(p[2]), "text": p[3],
                                     "type": p[4], "id": p[5] if len(p) > 5 else "-1"})
        if pmid:
            docs[pmid] = dict(pmid=pmid, title=title, abstract=abstract,
                              mentions=mentions, relations=relations)
    return docs

train = parse_pubtator(find_file("Training"))
dev   = parse_pubtator(find_file("Development"))
test  = parse_pubtator(find_file("Test"))
for n, d in [("train", train), ("dev", dev), ("test", test)]:
    cid = sum(len({(r["chemical"], r["disease"]) for r in x["relations"]}) for x in d.values())
    print(n, len(d), "articles,", cid, "CID relations")

In [ ]:
# CELL 3: TEXT PROCESSING
# - replace every entity mention by a placeholder (ENTxxx, letters only so spaCy tokenises "ENTAAB-induced" correctly)
# - delete parentheses that contain no entity
# - dependency-parse the simplified title and abstract with spaCy
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner"])

PH = re.compile(r"ENT[A-Z]{3}")
PAREN_NO_ENT = re.compile(r"\((?:(?!ENT[A-Z]{3})[^()])*\)")

def ph_name(n):
    return "ENT" + chr(65 + n // 676) + chr(65 + (n // 26) % 26) + chr(65 + n % 26)

def clean_mentions(mentions):
    out, last_end = [], -1
    for m in sorted(mentions, key=lambda m: (m["start"], -(m["end"] - m["start"]))):
        ids = [i.replace("MESH:", "") for i in m["id"].split("|") if i not in ("-1", "")]
        if not ids or m["start"] < last_end:        # skip unnormalised or overlapping mentions
            continue
        out.append(dict(m, ids=ids))
        last_end = m["end"]
    return out

def simplify(text, mentions, shift, counter, store):
    out, pos = [], 0
    for m in mentions:
        s, e = m["start"] - shift, m["end"] - shift
        if s < pos or s < 0 or e > len(text):
            continue
        name = ph_name(counter[0]); counter[0] += 1
        store[name] = m
        out += [text[pos:s], name]
        pos = e
    out.append(text[pos:])
    return PAREN_NO_ENT.sub("", "".join(out))

def build_sentences(spdoc, store, in_title):
    sents = []
    for sp in spdoc.sents:
        toks, base = list(sp), sp.start
        S = dict(text=[t.text for t in toks], lemma=[t.lemma_.lower() for t in toks],
                 tag=[t.tag_ for t in toks],
                 head=[min(max(t.head.i - base, 0), len(toks) - 1) for t in toks],
                 ent=[None] * len(toks), title=in_title)
        for i, t in enumerate(toks):
            mm = PH.search(t.text)
            if mm and mm.group() in store:
                S["ent"][i] = mm.group()
        sents.append(S)
    return sents

def process_corpus(docs):
    jobs = []
    for pmid, d in docs.items():
        ms, tlen = clean_mentions(d["mentions"]), len(d["title"])
        store, counter = {}, [0]
        t_text = simplify(d["title"], [m for m in ms if m["end"] <= tlen], 0, counter, store)
        a_text = simplify(d["abstract"], [m for m in ms if m["start"] >= tlen + 1], tlen + 1, counter, store)
        jobs.append((pmid, store, t_text, a_text))
    parsed = nlp.pipe([x for j in jobs for x in (j[2], j[3])], batch_size=64)
    out = {}
    for pmid, store, _, _ in jobs:
        t_doc, a_doc = next(parsed), next(parsed)
        out[pmid] = dict(sents=build_sentences(t_doc, store, True) + build_sentences(a_doc, store, False),
                         store=store,
                         gold={(r["chemical"], r["disease"]) for r in docs[pmid]["relations"]})
    return out

In [ ]:
# CELL 4: RELATION INSTANCE CONSTRUCTION with the four forms + filtering rules
#   Form 1: ARGi token* REL(verb)      token* ARGj
#   Form 2: ARGi token* REL(noun/verb) token* ARGj
#   Form 3: REL(noun) token* ARGi token* ARGj
#   Form 4: ARGi token* ARGj token* REL(noun/verb)
AUX = {"be", "have", "do"}
CONTEXT = 5            # how many tokens before ARGi / after ARGj we search for Forms 3 and 4
MAX_WORDS_CO = 10      # rule (1-i)
MAX_SENT_SPAN = 3      # rule (2-ii)
REQUIRE_REL = True     # drop instances where none of the four forms matches

def make_window(P, a, b):
    sents = P["sents"]
    W = dict(text=[], lemma=[], tag=[], ent=[], sid=[], off={}, sents=sents, store=P["store"])
    for k in range(a, b + 1):
        W["off"][k] = len(W["text"])
        for key in ("text", "lemma", "tag", "ent"):
            W[key] += sents[k][key]
        W["sid"] += [k] * len(sents[k]["text"])
    return W

def _verb(W, k):
    return W["tag"][k].startswith("VB") and W["lemma"][k] not in AUX and W["ent"][k] is None
def _noun(W, k):
    return W["tag"][k].startswith("NN") and W["ent"][k] is None

def find_rel(W, pi, pj):
    n = len(W["text"])
    f1 = [k for k in range(pi + 1, pj) if _verb(W, k)]
    f2 = [k for k in range(pi + 1, pj) if _noun(W, k) or _verb(W, k)]
    f3 = [k for k in range(max(0, pi - CONTEXT), pi) if _noun(W, k)]
    f4 = [k for k in range(pj + 1, min(n, pj + 1 + CONTEXT)) if _noun(W, k) or _verb(W, k)]
    n_rel = len(set(f1) | set(f2) | set(f3) | set(f4))
    if f1: return 1, f1[0], n_rel
    if f2: return 2, f2[0], n_rel
    if f3: return 3, f3[-1], n_rel
    if f4: return 4, f4[0], n_rel
    return 0, None, 0

def build_instances(processed, level_docs_gold=True):
    instances = []
    for pmid, P in processed.items():
        store, sents = P["store"], P["sents"]
        occ = [(si, ti, S["ent"][ti]) for si, S in enumerate(sents) for ti in range(len(S["text"])) if S["ent"][ti]]
        cum = np.cumsum([0] + [len(S["text"]) for S in sents])        # global token offsets

        def entity_pairs(o1, o2):
            m1, m2 = store[o1[2]], store[o2[2]]
            if m1["type"] == m2["type"]:
                return []
            c, d = (o1, o2) if m1["type"] == "Chemical" else (o2, o1)
            return [(ci, di, c, d) for ci in store[c[2]]["ids"] for di in store[d[2]]["ids"]]

        # ---- Co-occurrence level
        best = {}
        for a in range(len(occ)):
            for b in range(a + 1, len(occ)):
                o1, o2 = occ[a], occ[b]
                if o1[0] != o2[0]:
                    continue
                dist = abs(o1[1] - o2[1]) - 1
                if dist > MAX_WORDS_CO:
                    continue
                for ci, di, c, d in entity_pairs(o1, o2):
                    if (ci, di) not in best or dist < best[(ci, di)][0]:
                        best[(ci, di)] = (dist, o1, o2, c, d)
        co_pairs = set()
        for (ci, di), (dist, o1, o2, c, d) in best.items():
            W = make_window(P, o1[0], o1[0])
            pi, pj = o1[1], o2[1]
            form, rel, n_rel = find_rel(W, pi, pj)
            if REQUIRE_REL and form == 0:
                continue
            co_pairs.add((ci, di))
            instances.append(dict(pmid=pmid, level="CO", pair=(ci, di), W=W, pi=pi, pj=pj,
                                  form=form, rel=rel, n_rel=n_rel,
                                  c_name=c[2], d_name=d[2], label=int((ci, di) in P["gold"]),
                                  first=o1[2], second=o2[2]))
        # ---- Non-Co-occurrence level
        best = {}
        for a in range(len(occ)):
            for b in range(a + 1, len(occ)):
                o1, o2 = occ[a], occ[b]
                span = o2[0] - o1[0]
                if span < 1 or span > MAX_SENT_SPAN - 1:
                    continue
                dist = (cum[o2[0]] + o2[1]) - (cum[o1[0]] + o1[1])
                for ci, di, c, d in entity_pairs(o1, o2):
                    if (ci, di) in co_pairs:
                        continue
                    key = (span, dist)
                    if (ci, di) not in best or key < best[(ci, di)][0]:
                        best[(ci, di)] = (key, o1, o2, c, d)
        for (ci, di), (key, o1, o2, c, d) in best.items():
            W = make_window(P, o1[0], o2[0])
            pi, pj = W["off"][o1[0]] + o1[1], W["off"][o2[0]] + o2[1]
            form, rel, n_rel = find_rel(W, pi, pj)
            if REQUIRE_REL and form == 0:
                continue
            instances.append(dict(pmid=pmid, level="NCO", pair=(ci, di), W=W, pi=pi, pj=pj,
                                  form=form, rel=rel, n_rel=n_rel, c_name=c[2], d_name=d[2],
                                  label=int((ci, di) in P["gold"]), first=o1[2], second=o2[2]))
    return instances

In [ ]:
# CELL 5: FEATURE EXTRACTION (27 features, three categories: ctx_, dep_, stat_)
INC = {"increase", "elevate", "raise", "rise", "enhance", "augment", "high", "higher", "elevation"}
DEC = {"decrease", "reduce", "lower", "decline", "diminish", "attenuate", "reduction", "low", "drop"}
PEOPLE = {"infant", "adult", "patient", "child", "children", "subject", "volunteer", "man", "woman"}
DOSE = {"mg", "ml", "g", "kg", "mcg", "µg", "ug", "iu", "units", "mmol", "mg/kg"}

def dep_path(W, k):
    sid = W["sid"][k]; S = W["sents"][sid]; i = k - W["off"][sid]
    path, steps = [], 0
    while S["head"][i] != i and steps < 60:
        i = S["head"][i]; path.append(S["tag"][i]); steps += 1
    return "_".join(reversed(path)), steps

def near(W, k, words, width=3, attr="lemma"):
    lo, hi = max(0, k - width), min(len(W["text"]), k + width + 1)
    return int(any(W[attr][j] in words for j in range(lo, hi) if j != k))

def features(inst):
    W, store, pi, pj = inst["W"], inst["W"]["store"], inst["pi"], inst["pj"]
    first, second = inst["first"], inst["second"]
    pos = {W["ent"][k]: k for k in range(len(W["ent"])) if W["ent"][k] in (first, second)}
    pc = pos[inst["c_name"]]; pd_ = pos[inst["d_name"]]
    rel = inst["rel"]
    p1, h1 = dep_path(W, pi); p2, h2 = dep_path(W, pj)
    n = len(W["text"])
    ents = [e for e in W["ent"] if e]
    in_t_i = int(W["sents"][W["sid"][pi]]["title"]); in_t_j = int(W["sents"][W["sid"][pj]]["title"])
    between_verbs = sum(W["tag"][k].startswith("VB") for k in range(pi + 1, pj))
    f = {
        # contextual (1-3)
        "ctx_chem": store[inst["c_name"]]["text"].lower(),
        "ctx_dis": store[inst["d_name"]]["text"].lower(),
        "ctx_rel": W["lemma"][rel] if rel is not None else "NONE",
        # dependency (4-7)
        "dep_pos1": p1, "dep_pos2": p2, "dep_h1": h1, "dep_h2": h2,
        # statistical (8-27)
        "stat_n_rels": inst["n_rel"],
        "stat_verbs_sent": sum(t.startswith("VB") for t in W["tag"]),
        "stat_verbs_between": between_verbs,
        "stat_dist_cd": abs(pc - pd_),
        "stat_dist_i_rel": abs(rel - pi) if rel is not None else -1,
        "stat_dist_rel_j": abs(pj - rel) if rel is not None else -1,
        "stat_n_chem": sum(store[e]["type"] == "Chemical" for e in ents),
        "stat_n_dis": sum(store[e]["type"] == "Disease" for e in ents),
        "stat_n_ent": len(ents),
        "stat_i_title": in_t_i, "stat_j_title": in_t_j, "stat_both_title": in_t_i & in_t_j,
        "stat_dir_chem": near(W, pc, INC | DEC), "stat_dir_dis": near(W, pd_, INC | DEC),
        "stat_ard_chem": near(W, pc, PEOPLE), "stat_ard_dis": near(W, pd_, PEOPLE),
        "stat_dose_chem": near(W, pc, DOSE, 4, "text"),
        "stat_no_word": int(pj - pi == 1), "stat_one_word": int(pj - pi == 2),
        "stat_verb_between": int(between_verbs > 0),
    }
    return f

In [ ]:
# CELL 6: run text processing + instance construction + features for the 3 sets (a few minutes: spaCy parsing)
import time
t0 = time.time()
data = {}
for name, docs in [("train", train), ("dev", dev), ("test", test)]:
    P = process_corpus(docs)
    inst = build_instances(P)
    for i in inst:
        i["feat"] = features(i)
    gold_all = {(pm, c, d) for pm, x in P.items() for (c, d) in x["gold"]}
    data[name] = dict(inst=inst, gold=gold_all)
    co = [i for i in inst if i["level"] == "CO"]; nco = [i for i in inst if i["level"] == "NCO"]
    cover = len({(i["pmid"],) + i["pair"] for i in inst if i["label"]}) / len(gold_all)
    print(f"{name}: CO={len(co)} (pos {sum(i['label'] for i in co)})  NCO={len(nco)} (pos {sum(i['label'] for i in nco)})"
          f"  | gold relations reachable by candidates: {cover:.1%}  [{time.time()-t0:.0f}s]")

In [ ]:
# CELL 7: CLASSIFIERS  SVM + J48-like decision tree, averaged probabilities (WEKA "Vote": Average of Probabilities)
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

class LevelModel:
    def __init__(self, cats=("ctx", "dep", "stat")):
        self.cats = cats
        self.dv, self.sc = DictVectorizer(sparse=True), MaxAbsScaler()
    def _f(self, insts):
        return [{k: v for k, v in i["feat"].items() if k.split("_")[0] in self.cats} for i in insts]
    def fit(self, insts):
        X = self.sc.fit_transform(self.dv.fit_transform(self._f(insts)))
        y = [i["label"] for i in insts]
        # WEKA SMO: PolyKernel exponent 1 (=linear), C=0.6, buildCalibratorModels=True (Platt calibration)
        self.svm = SVC(kernel="linear", C=0.6, probability=True, random_state=0).fit(X, y)
        # J48 ~ C4.5: entropy criterion, min 2 instances per leaf, light pruning
        self.j48 = DecisionTreeClassifier(criterion="entropy", min_samples_leaf=2, ccp_alpha=0.001,
                                          random_state=0).fit(X, y)
        return self
    def proba(self, insts):
        X = self.sc.transform(self.dv.transform(self._f(insts)))
        s, j = self.svm.predict_proba(X)[:, 1], self.j48.predict_proba(X)[:, 1]
        return {"svm": s, "j48": j, "relSCAN": (s + j) / 2}

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return 100 * p, 100 * r, 100 * (2 * p * r / (p + r) if p + r else 0.0)

def evaluate(train_set, eval_set, cats=("ctx", "dep", "stat"), verbose=True):
    models = {lvl: LevelModel(cats).fit([i for i in data[train_set]["inst"] if i["level"] == lvl])
              for lvl in ("CO", "NCO")}
    ev = data[eval_set]["inst"]; gold = data[eval_set]["gold"]
    probs = {}
    for lvl in ("CO", "NCO"):
        sub = [i for i in ev if i["level"] == lvl]
        probs[lvl] = (sub, models[lvl].proba(sub))
    res = {}
    for algo in ("svm", "j48", "relSCAN"):
        predicted = set()
        for lvl in ("CO", "NCO"):
            sub, pr = probs[lvl]
            pred = pr[algo] >= 0.5
            y = np.array([i["label"] for i in sub])
            tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
            fn = int(((pred == 0) & (y == 1)).sum())
            res[(algo, lvl)] = prf(tp, fp, fn)
            predicted |= {(i["pmid"],) + i["pair"] for i, p in zip(sub, pred) if p}
        tp = len(predicted & gold); fp = len(predicted - gold); fn = len(gold - predicted)
        res[(algo, "FINAL")] = prf(tp, fp, fn)
    if verbose:
        print(f"train={train_set}  eval={eval_set}  features={cats}")
        print(f"{'algo':8s}{'level':7s}{'P':>7s}{'R':>7s}{'F1':>7s}")
        for (algo, lvl), (p, r, f1) in res.items():
            print(f"{algo:8s}{lvl:7s}{p:7.1f}{r:7.1f}{f1:7.1f}")
    return res

In [ ]:
# CELL 8: Setting 1 (train -> dev) and Setting 2 (train -> test)
_ = evaluate("train", "dev")
print()
_ = evaluate("train", "test")

In [ ]:
# CELL 9: feature-category ablation on the test set (paper's Table 5)
import itertools
for cats in [("ctx", "dep", "stat"), ("dep", "stat"), ("ctx", "stat"), ("ctx", "dep"),
             ("ctx",), ("dep",), ("stat",)]:
    r = evaluate("train", "test", cats, verbose=False)[("relSCAN", "FINAL")]
    print(f"{'+'.join(cats):14s} P={r[0]:5.1f}  R={r[1]:5.1f}  F1={r[2]:5.1f}")